In [1]:
import originpro as op 
import numpy as np 

In [8]:
def power_load(CT,omegaR=213.36,kappa=0.7,KP=1.0,sigma=0.1,Cd=0.01):
    ret=CT/omegaR*(CT**1.5/(2*np.sqrt(kappa))+KP*sigma*Cd/4.0)**-1
    return ret

def FM(CT,kappa,KP,sigma,Cd=0.01):
    ret=(CT**1.5/2)/(CT**1.5/2/np.sqrt(kappa)+KP*sigma*Cd/4.0)
    return ret 

In [13]:
op.attach()
wks=op.find_sheet("w","[Book1]Sheet1")


x=np.linspace(0,0.18,100)
sigma=0.1 
kappa=0.6
KP=0.7
CT=x*sigma
Cd=0.01

y1=power_load(CT=CT,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)
y2=FM(CT=CT,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)

wks.clear(0)
wks.from_list(0,x,lname=r"桨叶拉力系数 \q(CT/\sigma)")
wks.from_list(1,y1*1000,lname=r"功率载荷 \q(q/\rm{(N/kW)})")
wks.from_list(2,y2,lname=r"悬停效率 FM")

CT_max=(np.sqrt(kappa)*KP*sigma*Cd)**(2/3) 
FM_max=2/3*np.sqrt(kappa)
pl=power_load(CT=CT_max,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)
wks.from_list(3,CT_max/sigma,lname=r"最大 桨叶拉力系数 \q(CT/\sigma)")
wks.from_list(4,pl*1000,lname=r"最大 功率载荷 \q(q/\rm{(N/kW)})")
wks.from_list(5,FM_max,lname=r"最大 悬停效率 FM")

op.detach()

In [31]:
def fun(FM,kappa,Kt,Kp,sigma):
    ret=1/FM 
    ret=ret-1/np.sqrt(kappa)
    ret=ret/(2.6*Kp/((kappa*Kt)**1.5*np.sqrt(sigma)))
    ret=1/ret 
    return ret

op.attach()
wks=op.find_sheet("w","[Book2]Sheet1")

x=np.linspace(1,1.4,100)
kappa=1/x 
sigma=0.1
Kp=0.7
FM=0.75
Kt=0.734
y=fun(FM=FM,kappa=kappa,Kt=Kt,Kp=Kp,sigma=sigma)

wks.clear()
wks.from_list(col=0,data=x,lname=r"\q(1/\kappa)")
wks.from_list(col=1,data=y,lname=r"\q(C^{3/2}_{l}/C_{d})")

op.detach()

In [34]:
from NumOpt import Opti

opti=Opti()

x=1.15
kappa=1/x 
sigma=0.1
Kp=opti.variable(init_guess=0.7,freeze=False,lower_bound=0.0)
FM=0.75
Kt=opti.variable(init_guess=1.0,lower_bound=0.0)
y1=fun(FM=FM,kappa=1/1.125,Kt=Kt,Kp=Kp,sigma=sigma)
y2=fun(FM=FM,kappa=1/1.25,Kt=Kt,Kp=Kp,sigma=sigma)

opti.subject_to([
    y1==40.0,
    y2==100.0
])
opti.ipopt_solver()
sol=opti.solve()
print(sol(Kp))
print(sol(Kt))

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        4
Number of nonzeros in inequality constraint Jacobian.:        2
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        2
Total number of inequality constraints...............:        2
        inequality constraints with only lower bounds:        2
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 6.26e+01 1.00e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

RuntimeError: Error in Opti::solve [OptiNode] at .../casadi/core/optistack.cpp:217:
.../casadi/core/optistack_internal.cpp:1338: Assertion "return_success(accept_limit)" failed:
Solver failed. You may use opti.debug.value to investigate the latest values of variables. return_status is 'Infeasible_Problem_Detected'